# 02_01_DAOStarfinder

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, imexam, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed... now start install")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"****** module {pkg} is installed")
    else: 
        print(f"**** module {pkg} is installed")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module imexam is installed
**** module version_information is installed


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [2]:
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

This notebook was generated at 2023-02-01 19:52:04 (KST = GMT+0900) 
Ginga not installed, use other viewer, or no viewer
0 Python     3.8.16 64bit [GCC 11.2.0]
1 IPython    8.8.0
2 OS         Linux 5.15.0 58 generic x86_64 with glibc2.17
3 numpy      1.22.3
4 pandas     1.5.2
5 matplotlib 3.6.3
6 scipy      1.8.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 imexam     0.9.1
11 version_information 1.0.4


### import modules

In [7]:
import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import rcParams
from mpl_toolkits.axes_grid1 import make_axes_locatable

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import Python_utilities
import astro_utilities

from astropy.nddata import Cutout2D
from astropy.io import fits
from astropy.wcs import WCS

from photutils import DAOStarFinder
from photutils import IRAFStarFinder
from photutils.aperture import CircularAperture as CAp
from photutils.aperture import CircularAnnulus as CAn
from photutils import detect_threshold
from photutils.centroids import centroid_com
#from photutils import aperture_photometry as apphot

import warnings

from ccdproc import CCDData, ccd_process

from astropy.time import Time
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord

plt.rcParams.update({'figure.max_open_warning': 0})

/tmp/ipykernel_47686/2468915714.py:22: DeprecationWarning: `photutils.DAOStarFinder` is a deprecated alias for `photutils.detection.DAOStarFinder` and will be removed in the future. Instead, please use `from photutils.detection import DAOStarFinder` to silence this warning.
  from photutils import DAOStarFinder
/tmp/ipykernel_47686/2468915714.py:23: DeprecationWarning: `photutils.IRAFStarFinder` is a deprecated alias for `photutils.detection.IRAFStarFinder` and will be removed in the future. Instead, please use `from photutils.detection import IRAFStarFinder` to silence this warning.
  from photutils import IRAFStarFinder
/tmp/ipykernel_47686/2468915714.py:26: DeprecationWarning: `photutils.detect_threshold` is a deprecated alias for `photutils.segmentation.detect_threshold` and will be removed in the future. Instead, please use `from photutils.segmentation import detect_threshold` to silence this warning.
  from photutils import detect_threshold


In [8]:
#%%
BASEDIR = astro_utilities.base_dir

BASEDIRs = sorted(Python_utilities.getFullnameListOfsubDir(BASEDIR))
print ("BASEDIRs: {}".format(BASEDIRs))
print ("len(BASEDIRs): {}".format(len(BASEDIRs)))


BASEDIR = Path(BASEDIRs[0])
print ("Starting...\n{}".format(BASEDIR))

BASEDIR = Path(BASEDIR)

BASEDIR = Path(BASEDIR)
SOLVEDDIR = BASEDIR / astro_utilities.solved_dir2
DAORESULTDIR = BASEDIR / astro_utilities.DAOfinder_result_dir

if not DAORESULTDIR.exists():
    os.makedirs("{}".format(str(DAORESULTDIR)))
    print("{} is created...".format(str(DAORESULTDIR)))

BASEDIRs: ['/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-24_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-25_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-27_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-02_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-04_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-17_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/LANDOLT-114670_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/LANDOLT-114670_Light_-_2022-10-24_-_RiLA600_

In [9]:
#summary = yfu.make_summary(BASEDIR/"*.fit*")
summary = yfu.make_summary(SOLVEDDIR/"*.fit*")

if summary.empty:
        print("The dataframe(summary) is empty")
        pass
else:
    print("len(summary):", len(summary))
    print("summary:", summary)

All 123 keywords (guessed from /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_b_2022-10-23-10-58-26_180sec_RiLA600_STX-16803_-30C_2bin.fit) will be loaded.


/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key CDELT1 not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_b_2022-10-23-11-07-09_180sec_RiLA600_STX-16803_-30C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key CDELT2 not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_b_2022-10-23-11-07-09_180sec_RiLA600_STX-16803_-30C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key CROTA1 not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_b_2022-10-23-11-07-09_180sec_RiLA

len(summary): 64
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
1   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
2   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
3   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
4   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
..                                                ...       ...     ...   
59  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
60  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
61  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
62  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
63  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   

    BITPIX  NAXIS  NAXIS1  NAXIS2             FITS-TLM IMAGETYP  EXPOSURE

/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key PLTSOLVD not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_v_2022-10-23-13-59-45_120sec_RiLA600_STX-16803_-30C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key _CSAXES not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_v_2022-10-23-13-59-45_120sec_RiLA600_STX-16803_-30C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key _RPIX1 not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_v_2022-10-23-13-59-45_120sec_R

### Light

In [10]:
df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
df_light = df_light.reset_index(drop=True)
print("df_light:\n{}".format(df_light))


df_light:
                                                 file  filesize  SIMPLE  \
0   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
1   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
2   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
3   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
4   /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16845120    True   
..                                                ...       ...     ...   
59  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
60  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
61  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
62  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   
63  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...  16842240    True   

    BITPIX  NAXIS  NAXIS1  NAXIS2             FITS-TLM IMAGETYP  EXPOSURE  \
0      -32  

In [11]:
for _, row  in df_light.iterrows():
        fpath = Path(row["file"])
        print("type(fpath)", type(fpath))
        print("fpath", fpath)

        hdul = fits.open(fpath)
        hdr = hdul[0].header
        img = hdul[0].data
        print("img: {}".format(img))
        print("img.shape: {}".format(img.shape))

        # Set WCS and print for your information
        w = WCS(hdr)
        print("WCS: {}".format(w))

        thresh = detect_threshold(data=img, nsigma=3)
        thresh = thresh[0][0]
        print('detect_threshold', thresh)

        #%%
        try:
            FWHM   = 6

            DAOfind = DAOStarFinder(
                                    fwhm = FWHM, 
                                    threshold = thresh, 
                                    sharplo = 0.2, sharphi = 1.0,  # default values: sharplo=0.2, sharphi=1.0,
                                    roundlo = -1.0, roundhi = 1.0,  # default values -1 and +1
                                    sigma_radius = 1.5,           # default values 1.5
                                    ratio = 1.0,                  # 1.0: circular gaussian
                                    exclude_border = True         # To exclude sources near edges
                                    )
            # The DAOStarFinder object ("DAOfind") gets at least one input: the image.
            # Then it returns the astropy table which contains the aperture photometry results:
            DAOfound = DAOfind(img)
            print('{} star(s) founded by DAOStarFinder...'.format(len(DAOfound)))

            #%%
            if len(DAOfound)==0 :
                print ('No star was founded by DAOStarFinder...\n'*3)
            else : 

                # Use the object "found" for aperture photometry:
                N_stars = len(DAOfound)
                print('{} star(s) founded by DAOStarFinder...'.format(N_stars))
                DAOfound.pprint(max_width=1800)

                # save XY coordinates:
                DAOfound.write("{}/{}_DAOStarfinder_fwhm{}.csv".\
                                format(DAORESULTDIR, fpath.stem, FWHM), 
                                overwrite = True,
                                format='ascii.fast_csv')
                #%%
                print('type(DAOfound): {}'.format(type(DAOfound)))
                print('DAOfound: {}'.format(DAOfound))

                DAOcoord = np.array([DAOfound['xcentroid'], DAOfound['ycentroid']]).T
                print('type(DAOcoord): {}'.format(type(DAOcoord)))
                print('DAOcoord: {}'.format(DAOcoord))

                #%%
                # Save apertures as circular, 4 pixel radius, at each (X, Y)
                DAOapert = CAp((DAOcoord), r=4.)  
                print('type(DAOapert): {}'.format(type(DAOapert)))
                print('DAOapert: {}'.format(DAOapert))
                print('dir(DAOapert): {}'.format(dir(DAOapert)))

                DAOannul = CAn(positions = (DAOcoord), r_in = 4*FWHM, r_out = 6*FWHM) 
                print('type(DAOannul): {}'.format(type(DAOannul)))
                print('DAOannul: {}'.format(DAOannul))
                
                #%%
                plt.figure(figsize=(20,20))
                ax = plt.gca()

                ###########################################################
                # input some text for explaination. 
                plt.title("Result of DAOStarfinder", fontsize = 28, 
                    ha='center')

                plt.annotate('filename: {}'.format(fpath.stem), fontsize=10,
                    xy=(1, 0), xytext=(-500, -40), va='top', ha='left',
                    xycoords='axes fraction', textcoords='offset points')
                            
                plt.annotate('FWHM: {}'.format(FWHM), fontsize=10,
                    xy=(1, 0), xytext=(-1100, -30), va='top', ha='left',
                    xycoords='axes fraction', textcoords='offset points')
                    
                plt.annotate('Sky threshold: {:02f}'.format(thresh), fontsize=10,
                    xy=(1, 0), xytext=(-1100, -40), va='top', ha='left',
                    xycoords='axes fraction', textcoords='offset points')

                plt.annotate('Number of star(s): {}'.format(len(DAOfound)), fontsize=10,
                    xy=(1, 0), xytext=(-1100, -50), va='top', ha='left',
                    xycoords='axes fraction', textcoords='offset points')

                im = plt.imshow(img, 
                                vmin = thresh, 
                                vmax = thresh * 3,
                                #zscale=True,
                                origin='lower'
                                )

                DAOannul.plot(color='red', lw=2., alpha=0.4)
                
                divider = make_axes_locatable(ax)
                cax = divider.append_axes("right", size="3%", pad=0.05)
                plt.colorbar(im, cax=cax)

                plt.savefig(
                            "{}/{}_DAOStarfinder_fwhm{}.png".\
                                format(DAORESULTDIR, fpath.stem, FWHM)
                            )
                print("{}/{}_DAOStarfinder_fwhm{}.png is created...".\
                                format(DAORESULTDIR, fpath.stem, FWHM))
                #plt.show()
                plt.close() 

        except Exception as err:
            print('{0} with {1} '.format(err, fpath.name))


type(fpath) <class 'pathlib.PosixPath'>
fpath /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/solved2/KLEOPATRA_Light_b_2022-10-23-10-58-26_180sec_RiLA600_STX-16803_-30C_2bin.fit
img: [[7274.3535 7032.1226 7136.9775 ... 7007.1187 7080.712  6745.861 ]
 [7049.901  7102.915  7093.267  ... 7080.712  7020.0503 7237.7964]
 [7091.767  7123.0996 6956.123  ... 7142.93   7107.3887 7055.732 ]
 ...
 [7090.0234 7095.129  7104.7183 ... 7139.4585 7036.087  7062.32  ]
 [7071.743  7067.665  7131.3735 ... 7170.674  7054.696  6881.689 ]
 [7077.713  7167.15   7049.8145 ... 7115.597  7119.8423 7090.446 ]]
img.shape: (2048, 2048)
INFO: 
                Inconsistent SIP distortion information is present in the FITS header and the WCS object:
                SIP coefficients were detected, but CTYPE is missing a "-SIP" suffix.
                astropy.wcs is using the SIP distortion coefficients,
                therefore the coordinates calculated her

/home/guitar79/anaconda3/envs/astro_Python_ubuntu_env/lib/python3.8/site-packages/astropy/wcs/wcs.py:3064: RuntimeWarning: cdelt will be ignored since cd is present
  description.append(s.format(*self.wcs.cdelt))


detect_threshold 7299.290100097656
11 star(s) founded by DAOStarFinder...
11 star(s) founded by DAOStarFinder...
 id     xcentroid          ycentroid           sharpness       roundness1       roundness2     npix sky       peak              flux                 mag         
--- ------------------ ------------------ ------------------- ------------ ------------------- ---- --- ---------------- ------------------ ---------------------
  1  1407.349323352542 333.50846631561905 0.44770294886553297    0.0667913 0.18318282061155766   49 0.0   45931.90234375  8.740747451782227    -2.353871430643407
  2 1725.4439084151093  451.9085834271907  0.4856845727970591   0.18562917 0.20530978204997632   49 0.0   41677.37890625  7.527732849121094   -2.1916604951595615
  3 1774.6984519102905   627.599396195737  0.3006849288234841   0.19701272 0.02365406646175541   49 0.0 10974.5908203125 1.0278289318084717  -0.02980209561244404
  4 221.20130643334937  817.2949158221483  0.5667866214112863  -0.08539893 0.

/home/guitar79/anaconda3/envs/astro_Python_ubuntu_env/lib/python3.8/site-packages/astropy/wcs/wcs.py:3064: RuntimeWarning: cdelt will be ignored since cd is present
  description.append(s.format(*self.wcs.cdelt))


detect_threshold 7233.091178894043
12 star(s) founded by DAOStarFinder...
12 star(s) founded by DAOStarFinder...
 id     xcentroid          ycentroid           sharpness       roundness1       roundness2      npix sky       peak              flux                 mag         
--- ------------------ ------------------ ------------------- ------------ -------------------- ---- --- ---------------- ------------------ ---------------------
  1 1402.9420957364175  345.0182561625134  0.4402224646198867   0.18959971 -0.06348214295269358   49 0.0   47293.28515625  9.347763061523438   -2.4267692394093117
  2 1720.9598131385726  463.4640804753709  0.5079959332126246   0.27610016  0.06980304442196436   49 0.0   42807.91796875  7.588884353637695   -2.2004448367505995
  3 1770.1071335241832  639.1285060303578   0.559486143006675   0.23428456 0.051477945248954535   49 0.0 12745.6923828125  1.172786831855774  -0.17304770257823884
  4  216.6698825369593  828.4933417786708 0.42820573988060057 -0.0235159

...

detect_threshold 7229.032943725586
11 star(s) founded by DAOStarFinder...
11 star(s) founded by DAOStarFinder...
 id     xcentroid          ycentroid           sharpness      roundness1     roundness2     npix sky       peak              flux                mag         
--- ------------------ ------------------ ------------------- ---------- ------------------ ---- --- ---------------- ------------------ --------------------
  1  396.9447291423566 151.25311926553323  0.4297870919288259 0.24355897   0.50668142572101   49 0.0  18373.912109375 2.4245150089263916  -0.9615621923845141
  2  1379.709861592968  567.8493422046472 0.42364414433886016 0.50441074 0.3809915616171619   49 0.0   46615.76953125  8.321857452392578  -2.3005506808985317
  3 1697.3687387351258  687.1714371262412 0.46583607548928696  0.5725124 0.5610947897121492   49 0.0    42826.2734375 7.5172271728515625   -2.190144187383593
  4 1746.1153183976642  863.0413731368754  0.4777577839782705  0.5532899 0.5601830004450805   49 

detect_threshold 7038.212623596191
10 star(s) founded by DAOStarFinder...
10 star(s) founded by DAOStarFinder...
 id     xcentroid          ycentroid           sharpness      roundness1      roundness2     npix sky       peak              flux                 mag         
--- ------------------ ------------------ ------------------- ---------- ------------------- ---- --- ---------------- ------------------ ---------------------
  1   398.463770549856 176.54685055081953  0.3926395887112598 0.16102287   0.346944154472583   49 0.0 16294.2080078125  2.094045639038086   -0.8024653568795097
  2 1381.0579084368683  593.4163575446884 0.41014798209316417  0.5598149 0.46279633908061196   49 0.0   43206.07421875  8.062073707580566   -2.2661169109408643
  3 1698.8676138040296  712.9787060967225 0.41834091273373963  0.6019021  0.3784138743327066   49 0.0     38567.046875  6.917876243591309   -2.0999319712390268
  4  1747.486367011052  888.6578052722377 0.44935316951253423  0.4935002 0.457272484345

/home/guitar79/anaconda3/envs/astro_Python_ubuntu_env/lib/python3.8/site-packages/astropy/wcs/wcs.py:3064: RuntimeWarning: cdelt will be ignored since cd is present
  description.append(s.format(*self.wcs.cdelt))


detect_threshold 12421.030242919922
33 star(s) founded by DAOStarFinder...
33 star(s) founded by DAOStarFinder...
 id     xcentroid          ycentroid           sharpness       roundness1      roundness2      npix sky       peak             flux                 mag         
--- ------------------ ------------------ ------------------- ----------- -------------------- ---- --- --------------- ------------------ ---------------------
  1  788.2728144171509 126.22693308183483 0.47066291708302177  0.32408762 0.056410151747334596   49 0.0 19648.822265625 1.0090042352676392 -0.009732472949820277
  2 293.84535987182375   160.808955276851  0.4396673100223401   0.1734891  0.23130632554674757   49 0.0 23596.787109375 1.5455505847930908  -0.47270805946885863
  3  399.3865986888851  170.8850287410241 0.23303554043310173  0.24784137  0.36550617720417994   49 0.0  54265.41015625 5.4956841468811035   -1.8500544114713349
  4 1959.7463036954694 196.35116261481323  0.5317756781747338   0.6785726  0.0875

KeyboardInterrupt: 